# Rubik AI: training model besar di Google Colab (GPU)

Notebook ini melatih **CubeNet besar** (14,7 juta parameter, arsitektur setara paper DeepCubeA/EfficientCube)
memakai GPU gratis Colab, lalu mengujinya pada 100 kubus uji yang sama dengan eksperimen di laptop.

**Jalankan sel dari atas ke bawah.** Semua checkpoint disimpan di Google Drive, jadi kalau Colab
terputus (dan ini sering terjadi di versi gratis), cukup jalankan ulang semua sel: training otomatis
melanjutkan dari checkpoint terakhir.

## 1. Pastikan GPU aktif

Menu **Runtime → Change runtime type → T4 GPU → Save**. Sel di bawah harus menampilkan tabel berisi "Tesla T4".

In [ ]:
!nvidia-smi

## 2. Hubungkan Google Drive

Checkpoint disimpan ke folder `MyDrive/Rubik_AI_colab`. Izinkan akses saat diminta.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Rubik_AI_colab"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Folder penyimpanan:", DRIVE_DIR)

## 3. Ambil kode dari GitHub (repository private)

Repository private butuh **token akses**. Buat sekali di GitHub:

1. Foto profil → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**.
2. Isi nama (misalnya `colab-rubik`), pilih masa berlaku (misalnya 30 hari).
3. **Repository access → Only select repositories → Rubik_AI**.
4. **Permissions → Repository permissions → Contents → Read-only**.
5. Klik **Generate token**, salin token-nya.

Token hanya diketik di kotak yang muncul saat sel dijalankan. Token **tidak ditampilkan, tidak disimpan
di notebook, dan dihapus dari konfigurasi Git** setelah kode diunduh.

In [ ]:
import getpass, os, subprocess

REPO = "HaiqalGITuru/Rubik_AI"
WORKDIR = "/content/Rubik_AI"

if not os.path.exists(WORKDIR):
    token = getpass.getpass("Tempel GitHub token: ")
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/{REPO}.git", WORKDIR],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
    del token
else:
    print("Kode sudah ada di", WORKDIR)

os.chdir(WORKDIR)
!ls

## 4. Pengaturan training

- `STEPS = 100_000` dengan 1.000 scramble per batch (26.000 posisi per langkah).
  Perkiraan di T4: sekitar 0,1–0,2 detik per langkah, jadi **beberapa jam total**, kemungkinan besar
  terbagi dalam beberapa sesi Colab.
- **Jangan ubah `STEPS` setelah training dimulai**, karena jadwal learning rate (cosine) dihitung dari angka ini.

In [ ]:
MODEL_PATH = f"{DRIVE_DIR}/cubenet_large.pt"
STEPS = 100_000
SCRAMBLES = 1000
print("Checkpoint:", MODEL_PATH, "| ada:", os.path.exists(MODEL_PATH))

## 5. Training

Kecepatan (step/dtk) dan perkiraan sisa waktu tampil setiap 200 langkah. Checkpoint disimpan ke Drive
setiap 2.000 langkah. Untuk berhenti, klik tombol **stop** di sel ini: model langsung disimpan.

Kalau Colab terputus: jalankan ulang sel 2, 3, 4, lalu sel ini. Training melanjutkan otomatis.

In [ ]:
!python -m scripts.train --device cuda --hidden1 5000 --hidden2 1000 --blocks 4 --scrambles {SCRAMBLES} --steps {STEPS} --cosine --log-every 200 --eval-every 2000 --save {MODEL_PATH} --resume

## 6. Grafik training

Bisa dijalankan kapan saja, termasuk saat training belum selesai (setelah dihentikan).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(MODEL_PATH.replace(".pt", "_log.csv"))
ev = log.dropna(subset=["solve_d20"])

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(log.step, log.loss, color="#2453E8"); ax[0].set_title("Loss"); ax[0].set_xlabel("langkah")
ax[1].plot(log.step, log.accuracy, color="#2453E8"); ax[1].set_title("Akurasi"); ax[1].set_xlabel("langkah")
for d, c in [(10, "#A9B3C2"), (15, "#6B85E8"), (20, "#2453E8")]:
    ax[2].plot(ev.step, ev[f"solve_d{d}"] * 100, "o-", color=c, ms=3, label=f"jarak {d}")
ax[2].set_title("Tingkat berhasil (evaluasi ringan)"); ax[2].set_ylabel("%"); ax[2].legend(frameon=False)
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/grafik_training_large.png", dpi=150)
plt.show()

## 7. Ekspor model ringan

Checkpoint training sekitar 175 MB (terlalu besar untuk GitHub). Versi ini hanya berisi bobot model
dalam float16, sekitar **29 MB**, dengan prediksi yang sama. File ini yang nanti diunduh ke laptop.

In [ ]:
INFER_PATH = f"{DRIVE_DIR}/cubenet_large_infer.pt"
!python -m scripts.export_model {MODEL_PATH} {INFER_PATH}

## 8. Eksperimen beam pada 100 kubus uji yang sama

Hasil disimpan ke Drive. **Catatan untuk laporan:** waktu di sini diukur di GPU, jadi tidak bisa dibandingkan
langsung dengan waktu di laptop. Yang adil dibandingkan adalah **tingkat keberhasilan dan panjang solusi**.
Sel opsional di bawahnya menguji model kecil (laptop) di GPU yang sama, supaya perbandingan waktunya juga adil.

In [ ]:
import shutil
from IPython.display import Image, display

!python -m scripts.beam_experiment --model {INFER_PATH} --device cuda --name beam_experiment_large

shutil.copytree("results/beam_experiment_large", f"{DRIVE_DIR}/beam_experiment_large", dirs_exist_ok=True)
display(Image("results/beam_experiment_large/ringkasan.png"))

In [ ]:
# Opsional: model kecil (hasil training laptop) diuji di GPU yang sama, untuk perbandingan waktu yang adil
!python -m scripts.beam_experiment --model models/cubenet.pt --device cuda --name beam_experiment_small_gpu
shutil.copytree("results/beam_experiment_small_gpu", f"{DRIVE_DIR}/beam_experiment_small_gpu", dirs_exist_ok=True)

## 9. Bawa hasilnya ke laptop

Buka Google Drive di browser, folder **Rubik_AI_colab**, lalu unduh:

- `cubenet_large_infer.pt` → simpan ke `Rubik_AI/models/`
- `cubenet_large_log.csv` dan `grafik_training_large.png` → simpan ke `Rubik_AI/results/colab/`
- folder `beam_experiment_large` (dan `beam_experiment_small_gpu` jika dijalankan) → simpan ke `Rubik_AI/results/`

File `cubenet_large.pt` (175 MB) **tidak perlu** diunduh; biarkan di Drive untuk melanjutkan training kalau perlu.